## Import Packages

In [4]:
import cv2
import numpy as np
from dataloader import MOVE
import os
import random

## Global Variables

In [5]:
RANDOM_SAMPLE = False

### Helper Functions

In [2]:
def generate_path(rando : bool) -> str:
    if rando==True:
        ROOT = os.path.dirname(os.getcwd())
        DATA_DIR = os.path.join(ROOT, 'data')
        FIN_DIR = os.path.join(DATA_DIR, random.choice(list(filter(lambda x: os.path.isdir(os.path.join(DATA_DIR, x)), os.listdir(DATA_DIR)))))
        vp = os.path.join(FIN_DIR, random.choice(list(filter(lambda x: os.path.splitext(x)[1]=='.mov', os.listdir(FIN_DIR)))))
    else:
        vp = r'C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH16\TH16ST25A3D500TS0_2.mov'

    print(vp)

    return vp

## Dynamic Feature Detection
Pre-processing includes four steps
1. We need to load our *.mov* file in as our MOVE object. 
2. We convert to grayscale to reduce the information to process (and remove color variation)
3. Then, in preparation to "remove" the background from the video, we need to reduce the contrast
4. Lastly we need to remove the background

In [8]:
vid_path = generate_path(False)
recording = MOVE(vid_path)
#recording.play_video( )
recording.to_gray(inplace=True)


C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH16\TH16ST25A3D500TS0_2.mov


In [9]:
recording.change_contrast(alpha=2, beta=-10, inplace=True)
recording.remove_avg(skipframes=100)
recording.play_video()

In [6]:
#recording.play_video()

Here, we will create a copy (because we were too lazy to create an undo feature) to act as a checkpoint. We can optionally use a morphological transformation to try to fill out the air pockets as much as possible. We also check our work here at the end of this. We now have our mask for our dynamic components

In [7]:
out:MOVE = recording.ranger(minval=50, maxval=240)
out.median_blur(inplace=True)
#out.closing(inplace=True, ksize=(3,3), iterations=10)
#out.play_video()

### Error Checking

We can now check our mask

To ensure the accuracy of our mask, we will use one of the best, most robust vision systems: the human eye. We will overlay our mask to highlight the masked areas to a brighter level than the non-masked area. This is not super effective for spotting holes in bubble masks, but it is great for spotting missed bubbles.

In [8]:
#out.play_video()

In [ ]:
#out.show(out.frames[1000])

frame0 = out.frames[1000].copy()
h, w = frame0.shape[:2]
mask = np.zeros((h+2, w+2), np.uint8)
cv2.floodFill(frame0, mask, (0, 0), 255)
frame0 = cv2.bitwise_not(frame0)
frame0 = (frame0 | out.frames[1000])
#out.show(frame0)

# do flood fill
# im_floodfill = frame.copy()
# h, w = frame.shape[:2]
# mask = np.zeros((h+2, w+2), np.uint8)
# cv2.floodFill(im_floodfill, mask, (0, 0), 255)
# im_floodfill_inv = cv2.bitwise_not(im_floodfill)
# return (frame | im_floodfill_inv)

In [10]:
hm = out.hole_filler()
#hm.play_video()

In [11]:
#rec1 = MOVE(vid_path)
#rec1.apply_mask(out, inplace=True)
#rec1.play_video()

## Waterline Detection

In [12]:
recording = MOVE(vid_path)
recording.to_gray(inplace=True)
# vid_path = generate_path(True)

In [13]:
#recording.play_video()

In [14]:
background = recording.get_avg()
background = cv2.GaussianBlur(background, (1111, 1), sigmaX=32000, borderType=cv2.BORDER_REFLECT)
#MOVE.show(background)

In [15]:
Z = np.float32(background.reshape((-1, 3)))

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
_, labels, centers = cv2.kmeans(Z, 3, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

centers = np.uint8(centers)
quantized = centers[labels.flatten()]
quantized = quantized.reshape(background.shape)
#MOVE.show(quantized)

In [16]:
# figure out number/color mapping for each kmeans run

convenience = [centers[0][0], centers[1][0], centers[2][0]]

light_color = np.uint8(max(convenience))
convenience.remove(light_color)
mid_color = np.uint8(max(convenience))

#print(light_color)
#print(mid_color)

In [17]:
# ok we are doing this cs5 style

# PSEUDOCODE MOMENT

# take a "slice" in the middle of the clustered image
slice = quantized[:, quantized.shape[1]//2]
flatslice = slice.flatten()

# check that the slice is what you think it is
# for i in range(0, flatslice.size):
    # if flatslice[i] != quantized[i][quantized.shape[1]//2]:
        # print("AAAAAAH")

# scan down the slice and map the color transitions
# for i in range(0, flatslice.size - 1):
    # if flatslice[i] != flatslice[i + 1]:
        # print(flatslice[i], flatslice[i + 1])

# try to find the actual color transition we care about, from mid to light
waterline = 0
# for i in range(0, flatslice.size - 1):
    # if flatslice[i] == light_color and flatslice[i + 1] == mid_color:
        # waterline = i
        # print(waterline)
        # break

count = 0
i = 0
while count < 2 and i < flatslice.size:
    i += 1
    if flatslice[i] == mid_color and flatslice[i + 1] == light_color:
        count += 1
        waterline = i
        print(waterline)

336
544


In [18]:
# crop hole-filled image at waterline
frame1 = frame0[waterline:, :]
#MOVE.show(frame1)

**Messing with erosion/dilation**

In [19]:
frame2 = frame1.copy()
kernel = np.ones((10,10), np.uint8)
eroded = cv2.erode(frame2, kernel, iterations = 1)
#MOVE.show(frame1)
#MOVE.show(eroded)
dilated = cv2.dilate(eroded, kernel, iterations = 1)
#MOVE.show(dilated)

**Cropping video**

In [20]:
# crop video
hmm = hm.copy()
hmm = hmm.crop(row_start = waterline)
#hmm.play_video()


**PIPELINE OF EVIL**

In [ ]:
# give me frame n masked & cropped below the waterline
n = 400
frame_n = out.frames[n].copy()
h, w = frame_n.shape[:2]
mask = np.zeros((h+2, w+2), np.uint8)
cv2.floodFill(frame_n, mask, (0, 0), 255)
frame_n = cv2.bitwise_not(frame_n)
frame_n = (frame_n | out.frames[n])
#MOVE.show(frame_n)
cropped = frame_n[waterline:, :]
#MOVE.show(cropped)

In [22]:
#MOVE.show(cropped)

In [ ]:
# single-frame fin removal 

processed = cropped.copy()
kernel = np.ones((3,12), np.uint8)
#processed = cv2.dilate(processed, kernel10, iterations = 1)
processed = cv2.erode(processed, kernel, iterations = 1)
processed = cv2.dilate(processed, kernel, iterations = 1)
#processed = cv2.dilate(processed, kernel5, iterations = 1)
#MOVE.show(processed)

In [64]:
# helper function - returns frame n from the input video, black and white, cropped below the waterline

def frameN_cavity(vid, n):
    kernel = np.ones((3,12), np.uint8)
    frame_n = vid.frames[n].copy()

    h, w = frame_n.shape[:2]
    mask = np.zeros((h+2, w+2), np.uint8)
    cv2.floodFill(frame_n, mask, (0, 0), 255)
    frame_n = cv2.bitwise_not(frame_n)
    frame_n = (frame_n | out.frames[n])
    cropped = frame_n[waterline:, :]
    processed = cropped.copy()

    processed = cv2.erode(processed, kernel, iterations = 1)
    return cv2.dilate(processed, kernel, iterations = 1)

def frameN_fin(vid,  n):
    k1 = np.ones((2, 9), np.uint8)
    k2 = np.ones((12, 2), np.uint8)
    frame_n = vid.frames[n].copy()

    h, w = frame_n.shape[:2]
    mask = np.zeros((h+2, w+2), np.uint8)
    cv2.floodFill(frame_n, mask, (0, 0), 255)
    frame_n = cv2.bitwise_not(frame_n)
    frame_n = (frame_n | vid.frames[n])
    frame_n = frame_n[waterline:, :]
    frame_dupe = frame_n.copy()

    fin_removed = cv2.erode(frame_dupe, k1, iterations = 1)
    fin_removed = cv2.dilate(fin_removed, k1, iterations = 1)
    fin_removed = cv2.bitwise_not(fin_removed)

    fin = cv2.bitwise_and(fin_removed, frame_n)
    fin = cv2.erode(fin, k2, iterations = 1)
    return cv2.dilate(fin, k2, iterations = 1)

In [54]:
# air cavity retrieval (largest contour)
processed = frameN_cavity(out, 1100)

dupe = processed.copy()
MOVE.show(dupe)
contours, hierarchy = cv2.findContours(dupe, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
big = max(contours, key = cv2.contourArea)
blank = np.zeros(dupe.shape)
cv2.drawContours(blank, [big], 0, (255, 255, 255), 2)
MOVE.show(blank)

In [107]:
# air cavity retrieval (area threshold)
processed = frameN_cavity(out, 800)

dupe = processed.copy()
MOVE.show(dupe)
contours, hierarchy = cv2.findContours(dupe, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
big_contours = []
for contour in contours:
    if cv2.contourArea(contour) > 200:
        big_contours.append(contour)
blank = np.zeros(dupe.shape)
cv2.drawContours(blank, big_contours, -1, (255, 255, 255), 1)
MOVE.show(blank)

In [106]:
# fin retrieval (largest contour)
processed = frameN_fin(out, 937)

dupe = processed.copy()
MOVE.show(dupe)
contours, hierarchy = cv2.findContours(dupe, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
big = max(contours, key = cv2.contourArea)
blank = np.zeros(dupe.shape)
cv2.drawContours(blank, [big], 0, (255, 255, 255), 1)
MOVE.show(blank)

In [ ]:
hmmm = hmm.copy()
hmmm = hmmm.fin_removal(ksize = (3, 12))
#hmmm.play_video()

In [ ]:
hmmmm = hmm.copy()
hmmmm = hmmmm.fin_removal(ksize = (2, 9))
#hmmmm.play_video()

In [26]:
# fin removal 

big_bar = 255 * np.ones((100, hmm.shape[1]))
samples = np.uint16(np.random.randint(0, len(hmm.frames), (1, 10)))

for n in samples[0]:
    joined = np.concat((hmm.frames[n], big_bar, hmmm.frames[n]), axis = 0)
    #MOVE.show(joined)

In [27]:
# fin retrieval

big_bar = 255 * np.ones((100, hmm.shape[1]))
samples = np.uint16(np.random.randint(0, len(hmm.frames), (1, 10)))

for n in samples[0]:
    inverted = cv2.bitwise_not(hmmmm.frames[n])
    fin = cv2.bitwise_and(inverted, hmm.frames[n])
    kernel = np.ones((12,2), np.uint8)
    fin = cv2.erode(fin, kernel, iterations = 1)
    fin = cv2.dilate(fin, kernel, iterations = 1)
    joined = np.concat((hmm.frames[n], big_bar, fin), axis = 0)
    #MOVE.show(joined)

In [ ]:
hmmmmm = hmm.copy()
hmmmmm = hmmmmm.fin_retrieval(horiz_ksize = (2, 9), vert_ksize = (12, 2))
hmmmmm.play_video()

**END PIPELINE OF EVIL**

In [ ]:
yMin = quantized.shape[1]//4
darkestColor = np.unique(quantized).min()

half_mask:np.ndarray = np.ones_like(quantized)
half_mask[:yMin, :] = 0

In [ ]:
waterline = 255*np.where(quantized[:, ] == darkestColor, half_mask, np.zeros_like(recording.frames[0]))
MOVE.show(waterline)

In [ ]:
rec2 = MOVE(vid_path)
temp = MOVE(vid_path)
temp.img_like(img=waterline, inplace=True)
rec2.apply_mask(temp, inplace=True)
rec2.play_video()

In [ ]:
temp.combine_masks(out, inplace=True)
temp.play_video()

In [ ]:
for i in range(len(temp.frames)):
    final_frame = temp.frames[i]
    max_bright = final_frame.max()
    final_frame = final_frame.T
    for r, row in enumerate(final_frame):
        min_ind = np.where(row == max_bright)[0][0]
        final_frame[r][:min_ind] = 0

    temp.frames[i] = final_frame.T

temp.play_video()